In [ ]:
"""
Objective: Process open microdata from CFPB/NMDB, standardize mortgage delinquency 
by state, and render georeferenced choropleth maps using Plotly.
"""

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
import sys
import os

warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW MICRODATA PROCESSING (CFPB/NMDB)
# ==========================================
def process_spatial_dataset(raw_file: str, clean_file: str):
    print(f"[*] Processing CFPB microdata panel ({raw_file})...")
    
    try:
        df_raw = pd.read_csv(raw_file)
        required_cols = ['RegionType', 'Name', '2008-09']
        
        for col in required_cols:
            if col not in df_raw.columns:
                raise ValueError(f"Missing structural CFPB column: {col}")
                
        df_states = df_raw[df_raw['RegionType'] == 'State'].copy()
        
        state_mapping = {
            'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
            'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'District of Columbia': 'DC',
            'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL',
            'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA',
            'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN',
            'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
            'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
            'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
            'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
            'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
            'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
        }
        
        df_states['State_Code'] = df_states['Name'].map(state_mapping)
        df_clean = df_states[['State_Code', '2008-09']].rename(columns={'2008-09': 'delinq_mortgages_90d'}).dropna(subset=['State_Code'])
        
        df_clean.to_csv(clean_file, index=False)
        print(f"[*] Dataset processed. Dimension: {df_clean.shape[0]} states mapped.")
        
    except FileNotFoundError:
        print(f"[ERROR] File '{raw_file}' not found. Download it from the CFPB portal.")
        sys.exit(1)

# ==========================================
# 2. CHOROPLETH MAP RENDERING
# ==========================================
def render_maps(clean_file: str):
    print("\n[*] Rendering spatial maps...")
    
    if not os.path.exists(clean_file):
        print(f"[ERROR] Clean dataset '{clean_file}' is missing.")
        sys.exit(1)
        
    df_spatial = pd.read_csv(clean_file)
    
    portfolios = {
        'delinq_mortgages_90d': ('Mortgages (90+ days)', 'Reds'),
    }
    
    exported_maps = 0
    for col, (name, palette) in portfolios.items():
        if col not in df_spatial.columns:
            continue
            
        print(f" > Processing: {name}")
        
        # Layer 1: Choropleth (Heatmap)
        fig = px.choropleth(
            df_spatial, locations='State_Code', locationmode="USA-states",
            color=col, color_continuous_scale=palette, scope="usa",
            labels={col: 'Severe Delinquency (%)'}
        )
        
        # Layer 2: Scattergeo (State Abbreviations)
        fig.add_trace(go.Scattergeo(
            locations=df_spatial['State_Code'], locationmode="USA-states",
            text=df_spatial['State_Code'], mode="text",
            textfont=dict(color="black", size=10, family="Arial Black"), showlegend=False
        ))
        
        # Executive Layout
        fig.update_layout(
            geo=dict(bgcolor='rgba(0,0,0,0)', lakecolor='#e0f3ff', showland=True, landcolor='#f4f4f4'),
            margin=dict(l=0, r=0, t=20, b=0)
        )
        
        try:
            png_file = f'06_spatial_map_{col}.png'
            fig.write_image(png_file, scale=3)
            exported_maps += 1
        except Exception as e:
            print(f"[!] Failed to export PNG. Make sure the 'kaleido' package is installed. Error: {e}")
            
    print(f"\n[SUCCESS] Spatial Pipeline Completed. Maps rendered: {exported_maps}.")

if __name__ == "__main__":
    process_spatial_dataset('cfpb_nmdb_mortgages_2008Q3.csv', 'clean_spatial_states.csv')
    render_maps('clean_spatial_states.csv')